# siebel_id_origin - Build REL_ID to REGIE mapping

This notebook loads CSV files into Spark tables and computes `output` using historical relationship expansion.

Business rules implemented:
1. Seed REGIE clients only from `direct_bank` rows where `drc_bnk_f = 'Y'` and `edl_valid_to_dts = '9999-12-31 00:00:00.0000000'`.
2. Expand historically related `rel_id` values through both tables using identifier links:
   - `direct_bank`: `rel_id <-> np_sbl_id`
   - `ggm_np`: `rel_id <-> ikb_no`
3. Produce final pairs `(REL_ID, REL_ID_REGIE_KLANT)` with deterministic conflict handling (`min` seed if a record is reached by multiple seeds).

Each module displays intermediate DataFrames for easier inspection.

In [1]:
from pathlib import Path
import os
import re
import time

# Resolve project root no matter where the notebook is opened from.
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / 'spark_conf').exists() and (ROOT_DIR.parent / 'spark_conf').exists():
    ROOT_DIR = ROOT_DIR.parent

# Force the same runtime/config used by project scripts (important for local Hive metastore).
java_home = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'
os.environ['JAVA_HOME'] = java_home
os.environ['PATH'] = f"{java_home}/bin:" + os.environ.get('PATH', '')
os.environ['SPARK_CONF_DIR'] = str(ROOT_DIR / 'spark_conf')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

warehouse_dir = ROOT_DIR / '.local' / 'spark' / 'warehouse'
metastore_dir = ROOT_DIR / '.local' / 'spark' / 'metastore_db'
warehouse_dir.mkdir(parents=True, exist_ok=True)
metastore_dir.parent.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('siebel-id-rel-id-mapping')
    .config('spark.sql.catalogImplementation', 'hive')
    .config('spark.sql.warehouse.dir', str(warehouse_dir))
    .config('spark.hadoop.javax.jdo.option.ConnectionURL', f'jdbc:derby:;databaseName={metastore_dir};create=true')
    .config('spark.hadoop.javax.jdo.option.ConnectionDriverName', 'org.apache.derby.jdbc.EmbeddedDriver')
    .config('spark.hadoop.datanucleus.schema.autoCreateAll', 'true')
    .config('spark.hadoop.hive.metastore.schema.verification', 'false')
    .enableHiveSupport()
    .getOrCreate()
)

# Retry once if the embedded Derby metastore is briefly locked by another Spark process.
for attempt in range(2):
    try:
        spark.sql('CREATE DATABASE IF NOT EXISTS shebang')
        break
    except Exception as exc:
        message = str(exc)
        if 'Another instance of Derby' in message and attempt == 0:
            print('Metastore is temporarily locked; retrying in 3 seconds...')
            time.sleep(3)
            continue
        raise

input_dir = ROOT_DIR / 'use_cases' / 'siebel_id' / 'input'
if not input_dir.exists():
    raise FileNotFoundError(f'Input folder does not exist: {input_dir}')

csv_files = sorted(input_dir.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in: {input_dir}')

def to_table_name(path_obj: Path) -> str:
    base = path_obj.stem.lower()
    return re.sub(r'[^a-z0-9_]', '_', base)

created_tables = []
for csv_path in csv_files:
    table_name = to_table_name(csv_path)
    full_table = f'shebang.{table_name}'

    # Input files are tab-separated.
    df = (
        spark.read.option('header', True)
        .option('sep', '\t')
        .option('inferSchema', True)
        .csv(str(csv_path))
    )
    df.write.mode('overwrite').format('parquet').saveAsTable(full_table)

    row_count = spark.table(full_table).count()
    created_tables.append((csv_path.name, full_table, row_count))

print('Tables created/updated in shebang:')
for file_name, table, count in created_tables:
    print(f'- {file_name} -> {table} ({count} rows)')

tables_df = spark.sql('SHOW TABLES IN shebang')
display(tables_df.toPandas())

direct_bank_df = spark.table('shebang.direct_bank')
ggm_np_df = spark.table('shebang.ggm_np')

# Convert high-range timestamps to strings for friendly notebook rendering in pandas.
direct_bank_preview_df = (
    direct_bank_df
    .withColumn('edl_valid_from_dts', F.col('edl_valid_from_dts').cast('string'))
    .withColumn('edl_valid_to_dts', F.col('edl_valid_to_dts').cast('string'))
)
ggm_np_preview_df = (
    ggm_np_df
    .withColumn('edl_valid_from_dts', F.col('edl_valid_from_dts').cast('string'))
    .withColumn('edl_valid_to_dts', F.col('edl_valid_to_dts').cast('string'))
)

print('direct_bank sample:')
display(direct_bank_preview_df.limit(10).toPandas())

print('ggm_np sample:')
display(ggm_np_preview_df.limit(10).toPandas())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/18 18:18:15 WARN Utils: Your hostname, PM2745.local, resolves to a loopback address: 127.0.0.1; using 192.168.178.24 instead (on interface en0)
26/05/18 18:18:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 18:18:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/18 18:18:19 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/05/18 18:18:19 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore jguerrrero@

Tables created/updated in shebang:
- direct_bank.csv -> shebang.direct_bank (9 rows)
- ggm_np.csv -> shebang.ggm_np (112 rows)
- output.csv -> shebang.output (6 rows)


,namespace,tableName,isTemporary
0,shebang,direct_bank,False
1,shebang,ggm_np,False
2,shebang,output,False


direct_bank sample:


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
1,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
2,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
5,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
6,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
7,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y
8,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N


ggm_np sample:


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,105181982,2005-10-07 21:26:08,2017-07-29 19:45:56,1-6JEK8TE,N
1,115379534,2013-09-26 21:37:50,2017-07-29 19:09:15,1-6JEK8TE,N
2,115379534,2017-07-29 19:09:15,2017-10-10 16:09:06,1-6JEK8TE,N
3,105181982,2017-07-29 19:45:56,9999-12-31 00:00:00,1-6JEK8TE,Y
4,115379534,2017-10-10 16:09:06,2018-04-15 02:48:48.980578,1-6JEK8TE,N
5,118220032,2017-10-27 11:08:58,2017-11-01 16:56:44,1-1N09820A,N
6,118220032,2017-11-01 16:56:44,2017-11-02 14:43:54,1-1N09820A,N
7,118220032,2017-11-02 14:43:54,2017-12-06 16:02:23,1-1N09820A,N
8,118220032,2017-12-06 16:02:23,2019-05-11 03:32:32,1-1N09820A,N
9,115379534,2018-04-15 02:48:48.980578,2018-12-06 15:09:37,1-6JEK8TE,N


In [9]:
# Module 1: Identify REGIE seed clients from direct_bank based on active direct-bank rule.
# We match sentinel rows by prefix to avoid timestamp formatting differences.
seed_regie_df = (
    direct_bank_df
    .filter(
        (F.col('drc_bnk_f') == F.lit('Y'))
        & (F.col('edl_valid_to_dts').cast('string').startswith('9999-12-31'))
    )
    .select(F.col('rel_id').cast('string').alias('seed_rel_id'))
    .distinct()
    .orderBy('seed_rel_id')
)

print('Seed REGIE clients (REL_ID_REGIE_KLANT candidates):')
display(seed_regie_df.toPandas())

Seed REGIE clients (REL_ID_REGIE_KLANT candidates):


,seed_rel_id
0,118220032
1,118492640


## Module 2 - Build historical relationship edges

We build a unified historical edge set between `rel_id` and identifier keys from both source tables.
- `direct_bank`: `rel_id <-> np_sbl_id`
- `ggm_np`: `rel_id <-> ikb_no`

This edge set is used for transitive expansion.

In [7]:
edges_direct_df = (
    direct_bank_df
    .select(
        F.col('rel_id').cast('string').alias('rel_id'),
        F.col('np_sbl_id').cast('string').alias('id_key'),
    )
    .withColumn('source_table', F.lit('direct_bank'))
)

edges_ggm_df = (
    ggm_np_df
    .select(
        F.col('rel_id').cast('string').alias('rel_id'),
        F.col('ikb_no').cast('string').alias('id_key'),
    )
    .withColumn('source_table', F.lit('ggm_np'))
)

edges_df = (
    edges_direct_df
    .unionByName(edges_ggm_df)
    .filter(
        F.col('rel_id').isNotNull()
        & F.col('id_key').isNotNull()
        & (F.trim(F.col('rel_id')) != F.lit(''))
        & (F.trim(F.col('id_key')) != F.lit(''))
    )
    .distinct()
)

print('Historical edges count:', edges_df.count())
print('Sample edges:')
display(edges_df.orderBy('id_key', 'rel_id').limit(30).toPandas())

Historical edges count: 14
Sample edges:


,rel_id,id_key,source_table
0,117786942,1-1B89RM5A,direct_bank
1,115379534,1-1N09820A,ggm_np
2,118220032,1-1N09820A,direct_bank
3,118220032,1-1N09820A,ggm_np
4,11410074,1-1TWAZ5GR,ggm_np
5,117786942,1-1TWAZ5GR,ggm_np
6,118492640,1-1TWAZ5GR,direct_bank
7,118492640,1-1TWAZ5GR,ggm_np
8,105181982,1-6JEK8TE,ggm_np
9,115379534,1-6JEK8TE,ggm_np


## Module 3 - Transitive expansion from REGIE seeds

For each seed, we iteratively traverse the historical graph:
1. `rel_id -> id_key`
2. `id_key -> rel_id`

This computes the historical closure of related `rel_id` values per seed.

In [8]:
# Initialize traversal with self-mapping (each seed reaches itself).
frontier_df = seed_regie_df.select(
    F.col('seed_rel_id'),
    F.col('seed_rel_id').alias('rel_id'),
)
visited_df = frontier_df

max_iterations = 30
for iteration in range(1, max_iterations + 1):
    frontier_ids_df = (
        frontier_df.alias('f')
        .join(
            edges_df.alias('e'),
            F.col('f.rel_id') == F.col('e.rel_id'),
            'inner',
        )
        .select(F.col('f.seed_rel_id'), F.col('e.id_key'))
        .distinct()
    )

    candidate_rel_df = (
        frontier_ids_df.alias('i')
        .join(
            edges_df.alias('e'),
            F.col('i.id_key') == F.col('e.id_key'),
            'inner',
        )
        .select(F.col('i.seed_rel_id'), F.col('e.rel_id'))
        .distinct()
    )

    new_rel_df = (
        candidate_rel_df.alias('c')
        .join(
            visited_df.alias('v'),
            (F.col('c.seed_rel_id') == F.col('v.seed_rel_id'))
            & (F.col('c.rel_id') == F.col('v.rel_id')),
            'left_anti',
        )
    )

    new_count = new_rel_df.count()
    print(f'Iteration {iteration}: discovered {new_count} new rows')

    if new_count == 0:
        print('Traversal converged.')
        break

    visited_df = visited_df.unionByName(new_rel_df).distinct()
    frontier_df = new_rel_df

component_members_df = visited_df.orderBy('seed_rel_id', 'rel_id')

print('Component members discovered from each seed:')
display(component_members_df.toPandas())

Iteration 1: discovered 3 new rows
Iteration 2: discovered 1 new rows
Iteration 3: discovered 0 new rows
Traversal converged.
Component members discovered from each seed:


,seed_rel_id,rel_id
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032
3,118492640,11410074
4,118492640,117786942
5,118492640,118492640


## Module 4 - Build final output table

A `rel_id` could be reached by multiple seeds in dense historical graphs.
To keep output deterministic, we assign the smallest seed as `REL_ID_REGIE_KLANT`.

In [5]:
output_df = (
    component_members_df
    .groupBy('rel_id')
    .agg(F.min('seed_rel_id').alias('rel_id_regie_klant'))
    .select(
        F.col('rel_id').cast('long').alias('REL_ID'),
        F.col('rel_id_regie_klant').cast('long').alias('REL_ID_REGIE_KLANT'),
    )
    .orderBy('REL_ID')
)

print('Final output mapping:')
display(output_df.toPandas())

Final output mapping:


,REL_ID,REL_ID_REGIE_KLANT
0,11410074,118492640
1,105181982,118220032
2,115379534,118220032
3,117786942,118492640
4,118220032,118220032
5,118492640,118492640


## Module 5 - Optional validation against input/output.csv

If table `shebang.output` exists, this module compares computed output with expected output.

In [6]:
if spark.catalog.tableExists('shebang.output'):
    expected_df = (
        spark.table('shebang.output')
        .select(
            F.col('REL_ID').cast('long').alias('REL_ID'),
            F.col('REL_ID_REGIE_KLANT').cast('long').alias('REL_ID_REGIE_KLANT'),
        )
        .distinct()
        .orderBy('REL_ID')
    )

    missing_in_computed_df = expected_df.exceptAll(output_df)
    extra_in_computed_df = output_df.exceptAll(expected_df)

    print('Expected output:')
    display(expected_df.toPandas())

    print('Rows missing in computed output:')
    display(missing_in_computed_df.toPandas())

    print('Extra rows in computed output:')
    display(extra_in_computed_df.toPandas())
else:
    print("Table 'shebang.output' not found. Skipping validation module.")

Expected output:


,REL_ID,REL_ID_REGIE_KLANT
0,11410074,118492640
1,105181982,118220032
2,115379534,118220032
3,117786942,118492640
4,118220032,118220032
5,118492640,118492640


Rows missing in computed output:


,REL_ID,REL_ID_REGIE_KLANT


Extra rows in computed output:


,REL_ID,REL_ID_REGIE_KLANT
